## YouTube RAG Chatbot — Project

In [1]:
# --------------------------------------------------
# 1. Import Require Library
# --------------------------------------------------
import os

from dotenv import load_dotenv

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

c:\Users\kz\anaconda3\envs\GenAI\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.6.0)/charset_normalizer (3.5.1) doesn't match a supported version!
  warnings.warn(


In [2]:
# Step 2 — Load environment variables
# Load variables from .env
load_dotenv()

# Check that required API key exists
if not os.getenv("GROQ_API_KEY"):
    raise EnvironmentError(
        "GROQ_API_KEY is not configured in the .env file."
    )

### Step 3 — Load YouTube Transcript using YouTube API

- Key Note:
    - youtube-transcript-api is an open-source Python package and does not require a YouTube API key for its basic transcript-fetching functionality.

    - youtube-transcript-api is not the official YouTube Data API. It accesses transcript/subtitle information through an undocumented part of YouTube's web infrastructure.

- Install
    - pip install -q youtube-transcript-api



In [4]:
import youtube_transcript_api

print(youtube_transcript_api.__file__)
print(getattr(youtube_transcript_api, "__version__", "No __version__"))

c:\Users\kz\anaconda3\envs\GenAI\Lib\site-packages\youtube_transcript_api\__init__.py
No __version__


In [5]:
from youtube_transcript_api import YouTubeTranscriptApi

print(YouTubeTranscriptApi)
print(hasattr(YouTubeTranscriptApi, "fetch"))

<class 'youtube_transcript_api._api.YouTubeTranscriptApi'>
True


In [8]:
from youtube_transcript_api import YouTubeTranscriptApi

video_id = "FSZhPDzESPU"

ytt_api = YouTubeTranscriptApi()

transcript_data = ytt_api.fetch(
    video_id,
    languages=["en"]
)

transcript = " ".join(
    snippet.text for snippet in transcript_data
)

print(transcript)

RequestBlocked: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=FSZhPDzESPU! This is most likely caused by:

YouTube is blocking requests from your IP. This usually is due to one of the following reasons:
- You have done too many requests and your IP has been blocked by YouTube
- You are doing requests from an IP belonging to a cloud provider (like AWS, Google Cloud Platform, Azure, etc.). Unfortunately, most IPs from cloud providers are blocked by YouTube.

There are two things you can do to work around this:
1. Use proxies to hide your IP address, as explained in the "Working around IP bans" section of the README (https://github.com/jdepoix/youtube-transcript-api?tab=readme-ov-file#working-around-ip-bans-requestblocked-or-ipblocked-exception).
2. (NOT RECOMMENDED) If you authenticate your requests using cookies, you will be able to continue doing requests for a while. However, YouTube will eventually permanently ban the account that you have used to aut

In [2]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import RequestBlocked, TranscriptsDisabled

def get_transcript(video_id):
    try:
        data = YouTubeTranscriptApi().fetch(video_id, languages=["en"])
        return " ".join(s.text for s in data)
    except (RequestBlocked, TranscriptsDisabled):
        return get_transcript_ytdlp(video_id)

c:\Users\kz\anaconda3\envs\GenAI\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.6.0)/charset_normalizer (3.5.1) doesn't match a supported version!
  warnings.warn(


In [3]:
get_transcript("Gfr50f6ZBvo")

NameError: name 'get_transcript_ytdlp' is not defined

In [7]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled

video_id = "Gfr50f6ZBvo"

try:
    ytt_api = YouTubeTranscriptApi()

    transcript_data = ytt_api.fetch(
        video_id,
        languages=["en"]
    )

    transcript = " ".join(
        snippet.text for snippet in transcript_data
    )

    print(transcript)

except TranscriptsDisabled:
    print("No captions are available for this video.")

RequestBlocked: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=Gfr50f6ZBvo! This is most likely caused by:

YouTube is blocking requests from your IP. This usually is due to one of the following reasons:
- You have done too many requests and your IP has been blocked by YouTube
- You are doing requests from an IP belonging to a cloud provider (like AWS, Google Cloud Platform, Azure, etc.). Unfortunately, most IPs from cloud providers are blocked by YouTube.

There are two things you can do to work around this:
1. Use proxies to hide your IP address, as explained in the "Working around IP bans" section of the README (https://github.com/jdepoix/youtube-transcript-api?tab=readme-ov-file#working-around-ip-bans-requestblocked-or-ipblocked-exception).
2. (NOT RECOMMENDED) If you authenticate your requests using cookies, you will be able to continue doing requests for a while. However, YouTube will eventually permanently ban the account that you have used to aut

In [6]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled

video_id = "Gfr50f6ZBvo"  # Only the video ID, not the full URL

try:
    # Create API client
    ytt_api = YouTubeTranscriptApi()

    # Fetch English transcript
    transcript_data = ytt_api.fetch(
        video_id,
        languages=["en"]
    )

    # Convert transcript snippets into plain text
    transcript = " ".join(
        snippet.text for snippet in transcript_data
    )

    print(transcript)

except TranscriptsDisabled:
    print("No captions are available for this video.")

RequestBlocked: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=Gfr50f6ZBvo! This is most likely caused by:

YouTube is blocking requests from your IP. This usually is due to one of the following reasons:
- You have done too many requests and your IP has been blocked by YouTube
- You are doing requests from an IP belonging to a cloud provider (like AWS, Google Cloud Platform, Azure, etc.). Unfortunately, most IPs from cloud providers are blocked by YouTube.

There are two things you can do to work around this:
1. Use proxies to hide your IP address, as explained in the "Working around IP bans" section of the README (https://github.com/jdepoix/youtube-transcript-api?tab=readme-ov-file#working-around-ip-bans-requestblocked-or-ipblocked-exception).
2. (NOT RECOMMENDED) If you authenticate your requests using cookies, you will be able to continue doing requests for a while. However, YouTube will eventually permanently ban the account that you have used to aut